# 🤖 Notebook 04 — Modélisation ML
**Objectif** : Régression linéaire + Random Forest pour identifier les déterminants de la satisfaction
**Facteurs clés (issus de l'analyse R)** : Hygiene, Soins_Qualite, Reconfort, Competence_Personnel

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from src.models import (
    regression_lineaire, random_forest, predire_scenarios,
    plot_feature_importance, FEATURES_FINALES, FEATURES_COMPLETES
)

df = pd.read_csv('../data/processed/retraite_clean.csv')
print('✅ Données chargées')

## 1. Régression Linéaire Multiple
Reproduit le modèle final de l'analyse R :
`Satisfaction ~ Soins_Qualite + Competence_Personnel + Reconfort + Hygiene`

In [ ]:
res_lr = regression_lineaire(df, FEATURES_FINALES)

### Interprétation
- **Hygiène** : coefficient le plus élevé → levier n°1 sur la satisfaction
- **Soins** : deuxième facteur le plus impactant
- **Réconfort** : l'aspect humain pèse autant que les soins techniques
- **Compétence** : important mais légèrement inférieur aux trois précédents

Le R² (~15%) est cohérent avec l'analyse R (0.1562)

## 2. Visualisation des coefficients

In [ ]:
coeffs = res_lr['coefficients']

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if v < 0 else '#2980b9' for v in coeffs['Coefficient']]
ax.barh(coeffs['Variable'], coeffs['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coefficients de la régression linéaire', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient (impact sur la Satisfaction)')
plt.tight_layout()
plt.savefig('../reports/figures/coefficients_regression.png', dpi=150)
plt.show()

## 3. Random Forest — Toutes les variables

In [ ]:
res_rf = random_forest(df, FEATURES_COMPLETES, optimiser=False)

## 4. Importance des variables (Random Forest)

In [ ]:
plot_feature_importance(res_rf['importance'], '../reports/figures/feature_importance.png')

## 5. Comparaison des modèles

In [ ]:
comparaison = pd.DataFrame([
    {'Modèle': 'Régression Linéaire', 'MAE': res_lr['MAE'], 'RMSE': res_lr['RMSE'],
     'R²': res_lr['R2'], 'CV R²': res_lr['cv_r2_mean']},
    {'Modèle': 'Random Forest',       'MAE': res_rf['MAE'], 'RMSE': res_rf['RMSE'],
     'R²': res_rf['R2'], 'CV R²': res_rf['cv_r2_mean']},
])
print('── Comparaison des modèles ──')
comparaison

## 6. Prédictions — Scénarios (Partie 7 de l'analyse R)

In [ ]:
from sklearn.linear_model import LinearRegression

# Réentraîner sur tout le dataset
lr_final = LinearRegression()
lr_final.fit(df[FEATURES_FINALES], df['Satisfaction'])

scenarios = predire_scenarios(lr_final)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#f39c12', '#27ae60']
bars = ax.bar(scenarios['Profil'], scenarios['Satisfaction_prédite'],
              color=colors, edgecolor='white', width=0.4)
for bar, val in zip(bars, scenarios['Satisfaction_prédite']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}/10', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Satisfaction prédite par scénario', fontsize=13, fontweight='bold')
ax.set_ylabel('Note de satisfaction prédite (sur 10)')
ax.set_ylim(0, 11)
plt.tight_layout()
plt.savefig('../reports/figures/predictions_scenarios.png', dpi=150)
plt.show()
print('Profil Moyen (3/5 partout) : ~5.9/10')
print('Profil Excellent (5/5 partout) : ~8.7/10')

## 7. Conclusion
- Le **prix** n'influence pas la satisfaction (r=0.095, R²=0.9%)
- Les **4 piliers** de la satisfaction : Hygiène > Soins > Réconfort > Compétence
- Les **résidents Valides** sont les moins satisfaits (ANOVA, p<0.01)
- Améliorer tous les services de 2/5 à 5/5 fait gagner **~2.8 points** de satisfaction

In [ ]:
print('✅ Notebook 04 terminé — Modélisation complète !')
print('   → Voir les graphiques dans reports/figures/')
print('   → Voir les scripts R dans R/')